# Construcción de la arquitectura NPP-LSTM

## Objetivo
Reproducir la arquitectura del archivo NeuralPP.py
del proyecto de referencia.

Configuración:
- Historial: 20 eventos
- LSTM: 64 unidades
- Rama temporal: Softplus
- Rama de magnitud: ReLU y Sigmoid
- Umbral objetivo: Mw 4.5

Esta etapa construye la red, pero aún no la entrena.

In [10]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import tensorflow as tf

raiz = Path.cwd()
if not (raiz / "data").exists():
    raiz = raiz.parent

sys.path.insert(0, str(raiz / "src"))

from utils import datetime_to_hours, truncate_catalog_by_threshold
from npp_model import construir_npp

catalogo = pd.read_csv(
    raiz / "data/raw/SISMOS_LIMA_FORMATO_MODELO_cat.csv"
)

catalogo = datetime_to_hours(catalogo).dropna()
catalogo = truncate_catalog_by_threshold(catalogo, 3.5)

times = catalogo["time"].to_numpy()
mags = catalogo["mw"].to_numpy()

T_train = times[times < 450000]
M_train = mags[times < 450000]

# Mismas estadísticas del modelo original
mu_t = np.log(np.diff(T_train)).mean()
sigma_t = np.log(np.diff(T_train)).std()

mu_m = np.log(M_train).mean()
sigma_m = np.log(M_train).std()

print("Eventos de entrenamiento:", len(T_train))
print("Media log-tiempo:", mu_t)
print("Desviación log-tiempo:", sigma_t)
print("Media log-magnitud:", mu_m)
print("Desviación log-magnitud:", sigma_m)

Eventos de entrenamiento: 1621
Media log-tiempo: 4.143211780590066
Desviación log-tiempo: 2.16671713454715
Media log-magnitud: 1.5665146320262107
Desviación log-magnitud: 0.06987725623035673


In [11]:
modelo = construir_npp(
    ventana=20,
    mu_t=mu_t,
    sigma_t=sigma_t,
    mu_m=mu_m,
    sigma_m=sigma_m,
    m0pred=4.5
)

modelo.summary()

print("\nParámetros totales:", modelo.count_params())

Model: "NPP_LSTM_original"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ hist_t (InputLayer) │ (None, 20, 1)     │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ hist_m (InputLayer) │ (None, 20, 1)     │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lambda_12 (Lambda)  │ (None, 20, 1)     │          0 │ hist_t[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lambda_13 (Lambda)  │ (None, 20, 1)     │          0 │ hist_m[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tiempo_actual       │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_3       │ (None, 20, 2)     │          0 │ lambda_12[0][0],  │
│ (Concatenate)       │                   │            │ lambda_13[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ magnitud_actual     │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lambda_14 (Lambda)  │ (None, 1)         │          0 │ tiempo_actual[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm (LSTM)         │ (None, 64)        │     17,152 │ concatenate_3[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lambda_15 (Lambda)  │ (None, 1)         │          0 │ magnitud_actual[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_24 (Dense)    │ (None, 64)        │         64 │ lambda_14[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_25 (Dense)    │ (None, 64)        │      4,160 │ lstm[0][0]        │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_28 (Dense)    │ (None, 64)        │         64 │ lambda_15[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_29 (Dense)    │ (None, 64)        │      4,160 │ lstm[0][0]        │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_6 (Add)         │ (None, 64)        │          0 │ dense_24[0][0],   │
│                     │                   │            │ dense_25[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_7 (Add)         │ (None, 64)        │          0 │ dense_28[0][0],   │
│                     │                   │            │ dense_29[0][0],   │
│                     │                   │            │ dense_24[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_6        │ (None, 64)        │          0 │ add_6[0][0]       │
│ (Activation)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_7        │ (None, 64)        │          0 │ add_7[0][0]       │
│ (Activation)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_26 (Dense)    │ (None, 64)        │      4,160 │ activation_6[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_30 (Dense)    │ (None, 64)        │      4,160 │ activation_7[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_27 (Dense)    │ (None, 64)        │      4,160 │ dense_26[0][0]  

 Total params: 42,370 (165.51 KB)

 Trainable params: 42,306 (165.26 KB)

 Non-trainable params: 64 (256.00 B)


Parámetros totales: 42370


In [12]:
# Prueba de ejecución de la red sin entrenamiento

# Dos historiales de ejemplo, de 20 eventos cada uno
hist_t = tf.ones((2, 20, 1), dtype=tf.float32) * 24.0
hist_m = tf.ones((2, 20, 1), dtype=tf.float32) * 4.6

# Tiempo y magnitud que evaluaremos
dt = tf.ones((2, 1), dtype=tf.float32) * 24.0
mag = tf.ones((2, 1), dtype=tf.float32) * 5.0

# Pasar las entradas por la red
H, F = modelo(
    [hist_t, hist_m, dt, mag],
    training=False
)

print("Salida temporal H:", H.numpy())
print("Salida de magnitud F:", F.numpy())

print("Dimensión temporal:", H.shape)
print("Dimensión magnitud:", F.shape)

Salida temporal H: [[7.408909e-05]
 [7.408909e-05]]
Salida de magnitud F: [[0.10041054]
 [0.10041054]]
Dimensión temporal: (2, 1)
Dimensión magnitud: (2, 1)
